# Seed-language check: which related language should `guz_Latn` copy its embedding from?

**What this is.** `peft_setup.resize_model_for_new_language()` initializes the brand-new
`guz_Latn` token by literally copying one existing NLLB language token's embedding row into
it (`cfg["related_code"]`, currently `kik_Latn` in `arch_config.py`). That single line
decides the starting point every one of the 7 real experiments in `02_nllb_training.ipynb`
builds on - it's worth 20 minutes of zero-shot checking before spending A100-hours on all 7
with a choice that was never verified empirically for Ekegusii specifically.

**What this is not.** No training happens here - no `train_stage()` call, no gradient step.
For each candidate related-language, the base checkpoint is loaded fresh, resized, and the
new token's embedding is copied from that one candidate - then scored *zero-shot* (chrF2++,
same metric `metrics.py` treats as primary) on a small, fixed-seed subsample of the three
test sets. This isolates the effect of the embedding seed from everything else, since
nothing else about the model changes between candidates.

**Why these three candidates.** Ekegusii's actual closest relatives (Kuria, Logooli/Luhya,
Zanaki, Ikoma - Guthrie zone E40) aren't in NLLB-200 at all. Among what NLLB-200 does cover:
`kik_Latn` (Kikuyu, the current choice - Central Kenya/Thagicu Bantu, a different subgroup
from Gusii's), `lug_Latn` (Luganda - same broad Great Lakes/Interlacustrine Bantu zone
Gusii's own E40 cluster sits in, arguably a closer genealogical call than Kikuyu despite
being a different country), and `swh_Latn` (Swahili - genealogically the most distant of
the three, but NLLB's best-resourced African language by far, and the *other* direction
this project already has real Ekegusii training signal for). A comparable published case
(an unseen low-resource language added to an NLLB-family model) found single-proxy Swahili
outperform a multi-language-averaged initialization - resource-richness of the donor may
matter more than strict genealogical distance. This notebook checks whether that holds here
too, rather than assuming it from someone else's language pair.

**How to use the result:** whichever candidate wins on average, copy its code into
`arch_config.py`'s `"related_code"` value before running `02_nllb_training.ipynb`.

## Table of contents

1. Environment setup
2. A small, fixed-seed subsample of the three test sets
3. Candidate seed languages (with a validity check against this tokenizer)
4. Zero-shot generation and scoring, one fresh model load per candidate
5. Results and recommendation

**Where this runs.** Same project as `00_data_prep.ipynb`/`01_eda.ipynb`/
`02_nllb_training.ipynb` - run from anywhere under `Final_Training/` (e.g.
`Final_Training/Notebooks/`), reading `Final_Training/data/*.jsonl` and the shared
`.py` modules at `Final_Training/` root. Nothing here touches `raw_data/` - this
notebook only needs the already-built curriculum files, not the original 6 CSVs.

## 1. Environment setup

Same project-root convention as the other notebooks: walk up from wherever this
notebook actually sits (`Final_Training/Notebooks/`, per this project's layout) until
`arch_config.py` is found at `Final_Training/`, then treat that as project root. Only
`peft_setup`/`metrics`/`inference` are needed here - no `train.py` (nothing trains),
no `eda.py` (not an EDA notebook), no `data_io.load_curriculum`/`load_flat_splits`
(this only needs three specific `.jsonl` files, loaded directly with `data_io.load_jsonl`
in Section 2).

In [9]:
import sys, os, json, random
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, set_seed


def find_project_root(marker="arch_config.py"):
    current = Path.cwd().resolve()
    for candidate in [current] + list(current.parents):
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not find {marker} by walking up from {current}")


PROJECT_ROOT = find_project_root()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT))
print(f"project root: {PROJECT_ROOT}")

from arch_config import ARCH_CONFIG, MAX_LEN, SEED, DATA_DIR, LOGS_DIR
import data_io, peft_setup, metrics, inference

set_seed(SEED)
ARCHITECTURE = "nllb"
cfg = ARCH_CONFIG[ARCHITECTURE]
print(f"checkpoint       : {cfg['checkpoint']}")
print(f"current related_code (arch_config.py, unaffected by this notebook): {cfg['related_code']!r}")
print(f"GPU              : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'}")

project root: /home/jovyan/PSA
checkpoint       : facebook/nllb-200-distilled-600M
current related_code (arch_config.py, unaffected by this notebook): 'kik_Latn'
GPU              : NVIDIA H100 PCIe


## 2. A small, fixed-seed subsample of the three test sets

### Method
Up to `N_PER_CELL` rows per (test set, direction) combination - 3 test sets x 2 directions,
so at most `3 * 2 * N_PER_CELL` generation calls per candidate, not the full thousands-of-rows
evaluation `02_nllb_training.ipynb` runs for real. Sampling is seeded, so re-running this
notebook (or comparing a 4th candidate later) draws the identical subsample.

### Rationale
This check only needs enough signal to rank 3 candidates against each other, not
publication-grade precision on any single number - a few hundred generations per candidate
is a small fraction of a percent of the compute one real training run takes, which is the
entire point of doing this before committing to the real experiments rather than after.

In [10]:
N_PER_CELL = 100

def load_and_subsample(name, n, seed):
    rows = data_io.load_jsonl("Final_Training"/Path(DATA_DIR) / f"{name}.jsonl")
    by_direction = {}
    for direction in ["eng_Latn", "swh_Latn"]:
        subset = [r for r in rows if r["src_lang"] == direction]
        rng = random.Random(seed)
        by_direction[direction] = rng.sample(subset, min(n, len(subset)))
    return by_direction


test_samples = {name: load_and_subsample(name, N_PER_CELL, SEED)
                for name in ["test_psa", "test_bible", "test_general"]}

for name, by_dir in test_samples.items():
    for direction, rows in by_dir.items():
        print(f"  {name:14s} {direction:10s} {len(rows):>4,} rows sampled")


  test_psa       eng_Latn    100 rows sampled
  test_psa       swh_Latn    100 rows sampled
  test_bible     eng_Latn    100 rows sampled
  test_bible     swh_Latn    100 rows sampled
  test_general   eng_Latn    100 rows sampled
  test_general   swh_Latn    100 rows sampled


## 3. Candidate seed languages

### Method
Verify each candidate is an actual known token in this tokenizer before using it -
`tokenizer.convert_tokens_to_ids()` silently returns the `<unk>` id for a code that isn't a
real token rather than raising, which would make `resize_model_for_new_language()` copy the
`<unk>` embedding with no error at all. Cheap insurance worth having given `peft_setup.py`
itself doesn't check this.

### Rationale for the three candidates
See the intro: `kik_Latn` (current default), `lug_Latn` (same broad Bantu zone as Gusii,
arguably closer than Kikuyu), `swh_Latn` (least genealogically close, best-resourced, and
the language this project already trains a real Ekegusii direction from).

In [11]:
tokenizer = AutoTokenizer.from_pretrained(cfg["checkpoint"])
tokenizer = peft_setup.extend_tokenizer_for_new_language(tokenizer, cfg)

CANDIDATES = ["kik_Latn", "lug_Latn", "swh_Latn"]

unk_id = tokenizer.unk_token_id
for code_ in CANDIDATES:
    tok_id = tokenizer.convert_tokens_to_ids(code_)
    status = "OK" if tok_id != unk_id else "MISSING - resolves to <unk>, do not use"
    print(f"  {code_:10s} -> id {tok_id:>6}  [{status}]")
    assert tok_id != unk_id, f"{code_} is not a real token in this tokenizer"


Added new language token 'guz_Latn' at id 256204 (added=1)
  kik_Latn   -> id 256093  [OK]
  lug_Latn   -> id 256110  [OK]
  swh_Latn   -> id 256168  [OK]


## 4. Zero-shot generation and scoring, one fresh model load per candidate

### Method
For each candidate: load the base checkpoint fresh, resize its embeddings, initialize
`guz_Latn` by copying *that candidate's* embedding row (a local `cfg` copy with
`related_code` overridden - `arch_config.py` itself is never touched by this notebook), then
generate on the Section 2 subsample and score with `metrics.score()` - the same function
`02_nllb_training.ipynb` uses, so these chrF2++ numbers are directly comparable to that
notebook's zero-shot baseline row once you rerun it with the winning candidate.

### Rationale
One fresh model load per candidate, rather than swapping the embedding row in-place on a
single loaded model, avoids any risk of a previous candidate's forward-pass state or cache
leaking into the next candidate's numbers - the small extra load time is worth that
guarantee for a comparison this consequential.

In [12]:
def score_candidate(related_code):
    local_cfg = {**cfg, "related_code": related_code}
    model = AutoModelForSeq2SeqLM.from_pretrained(cfg["checkpoint"])
    model = peft_setup.resize_model_for_new_language(model, tokenizer, local_cfg)
    model.to("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    rows_out = []
    for test_name, by_dir in test_samples.items():
        for direction_code, lang_arg in [("eng_Latn", "en"), ("swh_Latn", "sw")]:
            subset = by_dir[direction_code]
            if not subset:
                continue
            with torch.no_grad():
                preds = [inference.generate_one(model, tokenizer, r["src"], lang_arg,
                                                 local_cfg, ARCHITECTURE, MAX_LEN)
                         for r in subset]
            refs = [r["tgt"] for r in subset]
            direction_label = "en-guz" if lang_arg == "en" else "sw-guz"
            score = metrics.score(preds, refs, f"seedcheck_{related_code}_{test_name}_{direction_label}")
            score.update({"related_code": related_code, "test_set": test_name.replace("test_", ""),
                         "direction": direction_label, "n": len(subset)})
            rows_out.append(score)

    del model
    torch.cuda.empty_cache()
    return rows_out


all_results = []
for candidate in CANDIDATES:
    print(f"\n{'='*70}\n  seeding guz_Latn from {candidate!r}\n{'='*70}")
    all_results.extend(score_candidate(candidate))



  seeding guz_Latn from 'kik_Latn'


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Resized embeddings 256206 -> 256205
Initialized 'guz_Latn' from 'kik_Latn'
New language ID: 256204
Related language ID: 256093
seedcheck_kik_Latn_test_psa_en-guz  chrF2++=13.36  BLEU=0.89
seedcheck_kik_Latn_test_psa_sw-guz  chrF2++=14.06  BLEU=1.94
seedcheck_kik_Latn_test_bible_en-guz chrF2++=14.74  BLEU=0.97
seedcheck_kik_Latn_test_bible_sw-guz chrF2++=13.75  BLEU=0.64
seedcheck_kik_Latn_test_general_en-guz chrF2++=15.28  BLEU=0.70
seedcheck_kik_Latn_test_general_sw-guz chrF2++=15.77  BLEU=1.08

  seeding guz_Latn from 'lug_Latn'


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Resized embeddings 256206 -> 256205
Initialized 'guz_Latn' from 'lug_Latn'
New language ID: 256204
Related language ID: 256110
seedcheck_lug_Latn_test_psa_en-guz  chrF2++=14.16  BLEU=1.23
seedcheck_lug_Latn_test_psa_sw-guz  chrF2++=15.42  BLEU=2.57
seedcheck_lug_Latn_test_bible_en-guz chrF2++=15.24  BLEU=0.25
seedcheck_lug_Latn_test_bible_sw-guz chrF2++=14.62  BLEU=0.18
seedcheck_lug_Latn_test_general_en-guz chrF2++=16.19  BLEU=0.98
seedcheck_lug_Latn_test_general_sw-guz chrF2++=16.48  BLEU=1.40

  seeding guz_Latn from 'swh_Latn'


Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]

Resized embeddings 256206 -> 256205
Initialized 'guz_Latn' from 'swh_Latn'
New language ID: 256204
Related language ID: 256168
seedcheck_swh_Latn_test_psa_en-guz  chrF2++=13.73  BLEU=1.71
seedcheck_swh_Latn_test_psa_sw-guz  chrF2++=14.76  BLEU=2.84
seedcheck_swh_Latn_test_bible_en-guz chrF2++=14.59  BLEU=0.83
seedcheck_swh_Latn_test_bible_sw-guz chrF2++=13.98  BLEU=0.27
seedcheck_swh_Latn_test_general_en-guz chrF2++=15.22  BLEU=1.53
seedcheck_swh_Latn_test_general_sw-guz chrF2++=16.47  BLEU=1.34


## 5. Results and recommendation

In [13]:
import pandas as pd

results_df = pd.DataFrame(all_results)[["related_code", "test_set", "direction", "n", "chrf2pp", "bleu"]]
print(results_df.to_string(index=False))

print("\nchrF2++ summary (mean across en/sw directions), per candidate x test set:")
summary = results_df.groupby(["related_code", "test_set"])["chrf2pp"].mean().unstack("test_set")
summary = summary[["bible", "psa", "general"]]
summary["overall_mean"] = summary.mean(axis=1)
print(summary.round(1).to_string())

winner = summary["overall_mean"].idxmax()
print(f"\nHighest mean zero-shot chrF2++: {winner!r} ({summary.loc[winner, 'overall_mean']:.1f})")
print(f"Next step: set arch_config.py's related_code to {winner!r} before running "
      f"02_nllb_training.ipynb's 7 real experiments.")

Path(LOGS_DIR).mkdir(exist_ok=True)
results_df.to_csv(f"{LOGS_DIR}/seed_language_check.csv", index=False)
summary.to_csv(f"{LOGS_DIR}/seed_language_check_summary.csv")

related_code test_set direction   n   chrf2pp     bleu
    kik_Latn      psa    en-guz 100 13.357359 0.893366
    kik_Latn      psa    sw-guz 100 14.059049 1.938220
    kik_Latn    bible    en-guz 100 14.739699 0.967672
    kik_Latn    bible    sw-guz 100 13.754828 0.640820
    kik_Latn  general    en-guz 100 15.284407 0.703797
    kik_Latn  general    sw-guz 100 15.773263 1.077301
    lug_Latn      psa    en-guz 100 14.161436 1.229217
    lug_Latn      psa    sw-guz 100 15.416744 2.565652
    lug_Latn    bible    en-guz 100 15.236996 0.252610
    lug_Latn    bible    sw-guz 100 14.616877 0.175316
    lug_Latn  general    en-guz 100 16.192660 0.977730
    lug_Latn  general    sw-guz 100 16.482600 1.402695
    swh_Latn      psa    en-guz 100 13.734690 1.708524
    swh_Latn      psa    sw-guz 100 14.760051 2.835292
    swh_Latn    bible    en-guz 100 14.589334 0.833985
    swh_Latn    bible    sw-guz 100 13.976351 0.268425
    swh_Latn  general    en-guz 100 15.217461 1.530953
    swh_La